# API レイテンシ計測ノートブック

ノートブック（Colab / ローカル Jupyter）から API を呼んだときの遅延を実測します。

**計測方法**（`benchmark/bench.py` と同じ）:
1. `GET /health` を複数回 → サーバ処理がほぼゼロなので **ネットワーク往復時間の基準値**
2. `POST /api/trade/next` を複数回 → 参加者ノートブックのループと同じ実ワークロード
3. 差分 ≒ サーバ内部の処理時間、として切り分けます

接続は `requests.Session()`（keep-alive）で使い回し、1回ごとの所要時間をすべて記録します。

In [ ]:
# ===== 設定 =====
# 計測先を選ぶ（コメントを付け替える）
BASE_URL = 'http://34.84.182.200:8000'                       # GCP 東京 VM（app+DB 同居）※動的IPのため要確認
# BASE_URL = 'https://app-production-7488.up.railway.app'    # Railway 本番
# BASE_URL = 'http://localhost:8000'                         # ローカル docker-compose

SCENARIO = 'DEMO_2016'
USER_ID  = 'testuser'
HEALTH_N = 20   # /health の計測回数
STEPS    = 30   # /api/trade/next の計測回数（上限）

In [ ]:
import statistics
import time

import requests

s = requests.Session()
s.get(f'{BASE_URL}/health', timeout=30)  # ウォームアップ（初回接続は計測に含めない）

# --- 1. /health ---
health_times = []
for _ in range(HEALTH_N):
    t0 = time.perf_counter()
    s.get(f'{BASE_URL}/health', timeout=30)
    health_times.append((time.perf_counter() - t0) * 1000)

# --- 2. セッションを作って /api/trade/next ---
r = s.post(f'{BASE_URL}/api/trade/start/{SCENARIO}/{USER_ID}', timeout=30)
r.raise_for_status()
session_id = r.json()['id']

body = {'session_id': session_id,
        'exchange_requests': [{'currency_from': 'JPY', 'currency_to': 'USD', 'amount': 1000}]}
step_times = []
for _ in range(STEPS):
    t0 = time.perf_counter()
    r = s.post(f'{BASE_URL}/api/trade/next', json=body, timeout=60)
    step_times.append((time.perf_counter() - t0) * 1000)
    if r.status_code != 200 or r.json().get('is_complete'):
        break

# --- 3. 集計 ---
def show(name, xs):
    print(f'{name:<22} n={len(xs):>3} 平均={statistics.mean(xs):>8.1f}ms '
          f'中央値={statistics.median(xs):>8.1f}ms 最小={min(xs):>8.1f}ms 最大={max(xs):>8.1f}ms')

print(f'計測先: {BASE_URL}\n')
show('GET /health', health_times)
show('POST /api/trade/next', step_times)

net = statistics.median(health_times)
step = statistics.median(step_times)
print(f'\nネットワーク往復（≒/health中央値）  : {net:.1f}ms')
print(f'サーバ内部処理（差分）              : {step - net:.1f}ms')
print(f'1000往復した場合の見込み            : {step:.1f}秒  (= 1ステップ{step:.1f}ms × 1000往復)')

## 結果の見方

- **`/health` の値 ≒ あなたの実行環境からサーバまでのネットワーク往復時間**。
  ローカル Jupyter（日本）→ GCP 東京 VM なら 5〜15ms、Colab（多くは米国リージョン）→ 東京 VM だと 100ms 超になります。
  **Colab で実行した場合に大きくなるのはこのためで、実行場所のリージョンが遅延を支配します。**
- **`trade/next − /health` ≒ サーバ内部の処理時間**。app と DB が同居している環境なら 5〜25ms 程度です。
  Railway 本番でこの値が数百 ms になるのは、app↔DB 間のネットワーク遅延（SQL 15往復分）が原因です（詳細は `benchmark/README.md`）。
- 参考実測値（2026-08-03）: Railway 本番 = 1ステップ約 1000ms / GCP 東京 VM（VM内から）= 約 22ms / ローカル同居 = 約 6ms